In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import pickle
import os

In [2]:
df = pd.read_csv('data/spotifydataset.csv')

df = df.drop(columns=['Unnamed: 0', 'artist_url'])
df['genres'] = df['genres'].fillna('unknown')
df = df.reset_index(drop=True)

print(df.shape)
df.head()

(1000, 21)


,artist_name,genres,followers,artist_popularity,track_name,album_name,release_date,duration_ms,explicit,track_popularity,...,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,Ariana Grande,pop,98934105,89,we can't be friends (wait for your love),eternal sunshine,2024-03-08,228639,False,89,...,0.646,5,-8.334,1,0.0427,0.0615,0.000030,0.0740,0.295,115.842
1,Ariana Grande,pop,98934105,85,the boy is mine,eternal sunshine,2024-03-08,173639,True,85,...,0.630,7,-5.854,0,0.0434,0.1570,0.000000,0.0732,0.447,97.998
2,Ariana Grande,pop,98934105,83,intro (end of the world),eternal sunshine,2024-03-08,92400,True,83,...,0.362,10,-9.480,1,0.0416,0.6700,0.000000,0.1760,0.385,84.726
3,Ariana Grande,pop,98934105,80,Save Your Tears (Remix) (with Ariana Grande) -...,After Hours (Deluxe),2020-03-20,191013,False,80,...,0.825,0,-4.645,1,0.0325,0.0215,0.000024,0.0936,0.593,118.091
4,Ariana Grande,pop,98934105,79,"yes, and?",eternal sunshine,2024-03-08,214994,True,79,...,0.775,1,-6.614,1,0.0548,0.1900,0.000065,0.1130,0.787,118.998


In [3]:
df['display_name'] = df['track_name'] + " — " + df['artist_name']
df['song_uid'] = df.index.astype(str)

FEATURE_COLS = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

X = df[FEATURE_COLS]
print("Null values:\n", X.isnull().sum())
print("\nFeature matrix shape:", X.shape)

Null values:
 danceability        0
energy              0
loudness            0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
dtype: int64

Feature matrix shape: (1000, 9)


In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling done!")
pd.DataFrame(X_scaled, columns=FEATURE_COLS).describe().round(2)

Scaling done!


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,-0.00,0.00,-0.00,0.00,-0.00,0.00,-0.00,-0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-3.24,-3.17,-7.22,-0.75,-0.92,-0.33,-1.12,-2.08,-2.60
25%,-0.63,-0.58,-0.26,-0.59,-0.82,-0.33,-0.64,-0.74,-0.79
50%,0.14,0.10,0.27,-0.40,-0.39,-0.33,-0.41,0.02,-0.00
75%,0.70,0.77,0.58,0.16,0.52,-0.33,0.40,0.73,0.59
max,2.11,1.63,1.66,8.66,2.53,3.98,5.47,1.96,2.84


In [6]:
k_values=[5,10,15,20]
metrics = ['cosine','euclidean','manhattan']
results = []

for k in k_values:
    for metric in metrics:
        knn_temp = NearestNeighbors(n_neighbors=k, metric=metric, algorithm='brute')
        knn_temp.fit(X_scaled)

        similarities = []
        test_indices = np.random.choice(len(df),50,replace=False)

        for idx in test_indices:
            distances,indices = knn_temp.kneighbors(
                X_scaled[idx].reshape(1,-1),n_neighbors=k+1
            )
            rec_features = X_scaled[indices[0][1:]]
            sim_matrix = cosine_similarity(rec_features)
            np.fill_diagonal(sim_matrix,0)
            avg_sim = sim_matrix.sum()/(k*(k-1))
            similarities.append(avg_sim)

        score = np.mean(similarities)
        results.append({'k': k, 'metric': metric, 'intra_similarity': score})
        print(f"k={k}, metric={metric} → {score:.4f}")



results_df = pd.DataFrame(results)
print("\nBest combination:")
print(results_df.loc[results_df['intra_similarity'].idxmax()])        

k=5, metric=cosine → 0.8406
k=5, metric=euclidean → 0.8404
k=5, metric=manhattan → 0.8010
k=10, metric=cosine → 0.7739
k=10, metric=euclidean → 0.7754
k=10, metric=manhattan → 0.7527
k=15, metric=cosine → 0.7364
k=15, metric=euclidean → 0.7290
k=15, metric=manhattan → 0.7093
k=20, metric=cosine → 0.7377
k=20, metric=euclidean → 0.7186
k=20, metric=manhattan → 0.6811

Best combination:
k                          5
metric                cosine
intra_similarity    0.840637
Name: 0, dtype: object


In [15]:
best = results_df.loc[results_df['intra_similarity'].idxmax()]

knn_model = NearestNeighbors(
    n_neighbors=20,
    metric = best['metric'],
    algorithm='brute'
)
knn_model.fit(X_scaled)
print(f"final model : k={int(best['k'])}, metric={best['metric']}")

final model : k=5, metric=cosine


In [16]:
song_index = pd.Series(df.index, index=df['song_uid'])

def recommend(song_uid, k=5):
    if song_uid not in song_index:
        return 'Song not found'
    
    idx = song_index[song_uid]
    
    distances, indices = knn_model.kneighbors(
        X_scaled[idx].reshape(1, -1), n_neighbors=k+1
    )
    
    rec_indices = indices[0][1:]
    rec_distances = distances[0][1:]
    
    recommendations = df.loc[rec_indices].copy()
    recommendations['similarity_score'] = 1 - rec_distances
    recommendations['final_score'] = (
        0.7 * recommendations['similarity_score'] +
        0.3 * recommendations['track_popularity'] / 100
    )
    
    return recommendations[
        ['display_name', 'album_name', 'track_popularity', 'final_score']
    ].sort_values('final_score', ascending=False).reset_index(drop=True)

In [17]:
print("Testing with:", df.loc[0, 'display_name'])
recommend("0")

Testing with: we can't be friends (wait for your love) — Ariana Grande


,display_name,album_name,track_popularity,final_score
0,Close To You — Gracie Abrams,The Secret of Us,85,0.884367
1,28 (feat. NiiHWA) — Agust D,D-2,70,0.876886
2,The Spectre — Alan Walker,The Spectre,72,0.865755
3,UNHEALTHY (feat. Shania Twain) — Anne-Marie,UNHEALTHY (Deluxe),63,0.811475
4,Sad (feat. Afrojack) — AFROJACK,Sad (feat. Afrojack),56,0.781791


In [18]:
# Catalog Coverage
recommended = set()
test_indices = np.random.choice(len(df), 200, replace=False)
for idx in test_indices:
    distances, indices = knn_model.kneighbors(
        X_scaled[idx].reshape(1, -1), n_neighbors=6
    )
    for i in indices[0][1:]:
        recommended.add(i)
print(f"Coverage: {len(recommended)/len(df)*100:.1f}% ({len(recommended)}/{len(df)} songs)")

# Hit Rate
hits = 0
sample = np.random.choice(len(df), 100, replace=False)
for idx in sample:
    distances, indices = knn_model.kneighbors(
        X_scaled[idx].reshape(1, -1), n_neighbors=6
    )
    rec_indices = indices[0][1:]
    if df.loc[idx, 'artist_name'] in df.loc[rec_indices, 'artist_name'].values:
        hits += 1
print(f"Hit Rate: {hits}%")

Coverage: 61.6% (616/1000 songs)
Hit Rate: 33%


In [19]:
os.makedirs('models', exist_ok=True)

# Save KNN model
with open('models/knn_model.pkl', 'wb') as f:
    pickle.dump(knn_model, f)

# Save scaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature matrix
with open('models/feature_matrix.pkl', 'wb') as f:
    pickle.dump(X_scaled, f)

# Save clean dataframe
df.to_csv('data/songs_clean.csv', index=False)

print("✅ knn_model.pkl saved")
print("✅ scaler.pkl saved")
print("✅ feature_matrix.pkl saved")
print("✅ songs_clean.csv saved")

✅ knn_model.pkl saved
✅ scaler.pkl saved
✅ feature_matrix.pkl saved
✅ songs_clean.csv saved
